# 06. Реальні фото: Arte-Lab і твої фото

**Тиждень 4.** Перевіряємо, чи працює на реальних фото те, що спрацювало на синтетиці.

**Arte-Lab** (EAN-13, CC BY 3.0; Zamberletti, Gallo, Albertini, ACPR 2013).
Завантаження: `python scripts/download_artelab.py`. Вміст кодів у датасеті не вказано,
тому правильна відповідь = те, що OpenCV і zxing-cpp **одностайно** прочитали на
оригіналі (`label_by_consensus`). Обмеження: набір зсувається в бік легких фото.

In [ ]:
import sys
from pathlib import Path

# ноутбук лежить у notebooks/, корінь проєкту на рівень вище
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

In [ ]:
from src.data import label_by_consensus, load_artelab, load_own_photos
from src.evaluate import run_benchmark, summarize

artelab = load_artelab()
labeled = label_by_consensus(artelab)
print("Фото:", len(artelab), "з консенсусом:", len(labeled))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, sample in zip(axes, labeled[:4]):
    ax.imshow(sample["image"], cmap="gray")
    ax.set_title(sample["text"], fontsize=9)
    ax.axis("off")
plt.show()

## Стенд пошкоджень на реальних фото

Беремо частину фото (для швидкості) і псуємо їх так само, як синтетику.

In [ ]:
SUBSET = 40   # TODO: збільш до всіх фото для фінальних цифр
subset = labeled[:SUBSET]
real_table = run_benchmark(subset, methods=["raw", "classic", "adaptive", "cascade"],
                           degradations=["dark", "uneven", "glare", "defocus", "noise"])
real_table.to_csv(RESULTS / "06_artelab.csv", index=False)
summarize(real_table, by=["method", "decoder"])

In [ ]:
summary = summarize(real_table[real_table["decoder"] == "zxing"], by=["method", "degradation"])
summary.pivot(index="degradation", columns="method", values="decode_rate").round(2)

## Твої фото

Як знімати: README, розділ «Власні фото». Тут немає штучних пошкоджень:
різне світло вже на самих фото (`normal`, `dark`, `glare`, `side`, `bright`).

In [ ]:
own = load_own_photos()
if own:
    # levels=[] -> лише "чисті" кадри: різне світло вже на самих фото
    own_table = run_benchmark(own, methods=["raw", "classic", "adaptive", "cascade"], levels=[])
    own_info = pd.DataFrame([{"sample": s["id"], "light": s["light"], "package": s["package"]} for s in own])
    own_table = own_table.merge(own_info, on="sample")
    own_table.to_csv(RESULTS / "06_own.csv", index=False)
    print(summarize(own_table, by=["light", "method"]).pivot(index="light", columns="method", values="decode_rate").round(2))
else:
    print("Власних фото ще немає")

**TODO для тебе:**
- Чи збігається ранжування методів на реальних фото і на синтетиці?
- Які упаковки (`package`) найважчі? Чи підтверджується, що глянець гірший за картон?